# Day 11: Quantization — Number Formats (FP8, INT8, INT4)
> *Inference Engineering* — Chapter 5.1.1 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 04 (Arithmetic Intensity)


## Concept Overview

Quantization reduces model precision from FP16/BF16 to lower-bit formats to reduce memory footprint and increase arithmetic throughput. FP8 (E4M3 or E5M2) halves memory vs FP16 with minimal accuracy loss on modern hardware (H100+). INT8 is widely supported and roughly doubles throughput vs FP16. INT4 is the sweet spot for weight-only quantization — 4x compression at acceptable accuracy loss. The key tradeoff: lower bits = less memory + faster compute, but higher quantization error requires calibration or training-aware compensation.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Number Format Ranges and Precision

Different formats trade off between representable range and precision (mantissa bits).

| Format | Bits | Exp | Mantissa | Max Value | Min Normal |
|--------|------|-----|----------|-----------|------------|
| FP32   | 32   | 8   | 23       | 3.4e38    | 1.2e-38    |
| BF16   | 16   | 8   | 7        | 3.4e38    | 1.2e-38    |
| FP16   | 16   | 5   | 10       | 65504     | 6.1e-5     |
| FP8 E4M3 | 8  | 4   | 3        | 448       | 0.0156     |
| FP8 E5M2 | 8  | 5   | 2        | 57344     | 1.5e-5     |
| INT8   | 8    | -   | -        | 127       | -          |
| INT4   | 4    | -   | -        | 7         | -          |


In [ ]:
def quantize_symmetric(x, n_bits, scale=None):
    """Symmetric uniform quantization."""
    qmax = 2**(n_bits-1) - 1
    if scale is None:
        scale = x.abs().max() / qmax
    x_q = torch.clamp(torch.round(x / scale), -qmax, qmax).to(torch.int8)
    x_dq = x_q.float() * scale
    return x_q, x_dq, scale

# Generate a realistic weight distribution
torch.manual_seed(42)
weights = torch.randn(1000) * 0.02  # typical transformer weight scale

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, bits, title in zip(axes, [8, 4, 2], ['INT8', 'INT4', 'INT2']):
    _, dq, scale = quantize_symmetric(weights, bits)
    error = (weights - dq).abs()
    ax.hist(weights.numpy(), bins=50, alpha=0.6, label='Original', density=True)
    ax.hist(dq.numpy(), bins=50, alpha=0.6, label=f'{title} dequant', density=True)
    ax.set_title(f'{title}: scale={scale:.4f}, MAE={error.mean():.4f}')
    ax.legend(fontsize=8)
    ax.set_xlabel('Weight value')

plt.tight_layout()
plt.savefig('quantization_formats.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved quantization_formats.png')


## 2. Quantization Error vs Bit Width

The quantization error scales with $\Delta/2$ where $\Delta = \text{range}/(2^N - 1)$ is the step size. For Gaussian weights, the expected MSE under uniform quantization is approximately $\Delta^2/12$.


In [ ]:
results = []
for bits in [1, 2, 3, 4, 5, 6, 7, 8]:
    _, dq, _ = quantize_symmetric(weights, bits)
    mse = ((weights - dq)**2).mean().item()
    snr = (weights**2).mean().item() / (mse + 1e-12)
    results.append({'bits': bits, 'mse': mse, 'snr_db': 10*np.log10(snr)})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
bits_list = [r['bits'] for r in results]
mse_list = [r['mse'] for r in results]
snr_list = [r['snr_db'] for r in results]

ax1.semilogy(bits_list, mse_list, 'b-o')
ax1.set_xlabel('Bit Width'); ax1.set_ylabel('MSE')
ax1.set_title('Quantization Error vs Bit Width')
ax1.set_xticks(bits_list)
ax1.grid(True)

ax2.plot(bits_list, snr_list, 'r-o')
ax2.set_xlabel('Bit Width'); ax2.set_ylabel('SNR (dB)')
ax2.set_title('Signal-to-Noise Ratio vs Bit Width')
ax2.set_xticks(bits_list)
ax2.axhline(20, color='gray', linestyle='--', label='20dB threshold')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('quantization_snr.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'{"Bits":>6} {"MSE":>12} {"SNR (dB)":>10}')
for r in results:
    print(f'{r["bits"]:>6} {r["mse"]:>12.6f} {r["snr_db"]:>10.1f}')


## 3. Memory and Throughput Savings

The practical impact: how much GPU memory does quantization save, and how does this translate to throughput?


In [ ]:
model_configs = [
    ('Llama-3-8B',   8e9),
    ('Llama-3-70B',  70e9),
    ('Llama-3-405B', 405e9),
]
formats = [
    ('FP32', 4), ('BF16', 2), ('FP16', 2),
    ('FP8',  1), ('INT8', 1), ('INT4', 0.5),
]
print(f'{"Model":<20} {"FP32 (GB)":>10}', end='')
for fmt, _ in formats[3:]:
    print(f' {fmt+" (GB)":>9}', end='')
print()
print('-'*80)
for name, params in model_configs:
    fp32_gb = params * 4 / 1e9
    print(f'{name:<20} {fp32_gb:>10.0f}', end='')
    for fmt, bytes_per in formats[3:]:
        gb = params * bytes_per / 1e9
        print(f' {gb:>9.0f}', end='')
    print()

print('\nThroughput multipliers vs FP16 (approximate, A100/H100):')
for fmt, throughput_mult in [('FP16',1.0),('FP8',1.9),('INT8',2.0),('INT4',3.5),('INT4 w/o dequant',4.0)]:
    print(f'  {fmt:<25} {throughput_mult:.1f}x')


## Experiments: Try These

1. **Outlier impact**: Generate weights with 1% extreme outliers and compare INT8 vs INT4 quantization error. This motivates SmoothQuant and GPTQ.
2. **Block quantization**: Implement block-wise quantization (one scale per 128 elements) and compare error to per-tensor quantization.
3. **FP8 simulation**: Implement E4M3 quantization in Python using float32 arithmetic. Verify max value = 448.


## Key Takeaways

- Quantization maps floating-point weights/activations to integers, trading precision for memory and compute savings.
- FP8 E4M3 is the modern sweet spot for activations: same range as BF16, 2x memory saving, natively supported on H100/Blackwell.
- INT4 weight-only quantization (4 bits for weights, FP16 for activations) gives 4x memory savings at ~1-2% perplexity cost for large models.
- Quantization error scales as $O(2^{-N})$ — each additional bit halves the error, but hardware throughput gains get smaller above 8 bits.

## References
- *Inference Engineering* Ch 5.1.1 — Philip Kiely, Baseten Books 2026
- Dettmers et al. (2022), "LLM.int8(): 8-bit Matrix Multiplication for Transformers"
- NVIDIA FP8 formats specification (H100)
